In [8]:
import nest_asyncio
nest_asyncio.apply()
from dotenv import load_dotenv
import os

# from telegram import Update
# from telegram.ext import ApplicationBuilder, CommandHandler, ContextTypes

from rag_retrieve import retrieve_5_most_relevant

from pydantic_ai import Agent
from pydantic_ai.models.openrouter import OpenRouterModel
from pydantic_ai.providers.openrouter import OpenRouterProvider

# Load environment variables from .env file
load_dotenv()
TELEGRAM_BOT_TOKEN = os.getenv('TELEGRAM_BOT_TOKEN', '')
OPENROUTER_API_KEY = os.getenv('OPENROUTER_API_KEY', '')

model = OpenRouterModel(
    'deepseek/deepseek-chat',
    provider=OpenRouterProvider(api_key=OPENROUTER_API_KEY),
)

agent = Agent(
    model,
    system_prompt=(
        "You are Niccolò Machiavelli. Answer in this exact structure:\n"
        "1. Citation from Machiavelli's The Prince.\n"
        "2. Interpretation of the citation.\n"
        "3. Direct steps the user should take.\n"
        "Use the provided excerpts as your source material. Answer in Russian."
    ),
)

def ask_advice(user_input):
    passages = retrieve_5_most_relevant(user_input)

    context = "\n\n".join(
        f"{chapter}\n{paragraph}" for chapter, paragraph in passages
    )

    prompt = f"""Situation:
    {user_input}

    Relevant excerpts from The Prince:
    {context}
    """

    return agent.run_sync(prompt).output


In [10]:
ask_advice("Как поступать с приятелями, которые говорят про тебя за спиной плохие вещи?")

'1. Цитата из «Государя»:  \n«Таким образом, государь всегда должен советоваться с другими, но только когда он того желает, а не когда того желают другие; и он должен осаживать всякого, кто вздумает, непрошеный, подавать ему советы.»\n\n2. Интерпретация:  \nМакиавелли подчеркивает важность контроля над тем, кто и когда может давать советы. Он советует государям быть избирательными в общении и не позволять другим влиять на их решения без их согласия. Это помогает избежать манипуляций и сохранить авторитет.\n\n3. Прямые шаги для пользователя:  \n- Ограничьте круг людей, которым вы доверяете мнение, теми, кто доказал свою мудрость и честность.  \n- Не реагируйте на непрошеные советы или сплетни, особенно если они исходят от людей, не заслуживающих доверия.  \n- Действуйте решительно и держитесь своего мнения, чтобы не потерять уважение окружающих.  \n- Если вам необходимо узнать правду, задавайте конкретные вопросы и слушайте ответы, но окончательное решение всегда принимайте самостоятель